# Graph Tester: evaluación de modelos de segmentación

Este notebook reúne las pruebas finales utilizadas para comprobar el rendimiento de los modelos de segmentación del TFG.

El análisis compara las máscaras generadas automáticamente por los modelos con las segmentaciones manuales (*ground truth*) y calcula métricas cuantitativas y visuales para las regiones cerebrales y de lesión isquémica.

In [ ]:
#Ejecutar solo si el entorno no dispone de estas librerías.
#!pip install tensorflow
#!pip install plotly pandas scikit-learn scikit-image

# Carga del dataset ground truth

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import re

# --- CONFIG ---
DATASET_DIR = "dataset"

# Función para cargar volumen NIfTI
def cargar_volumen_nifti(nifti_path):
    nii = nib.load(str(nifti_path))
    vol = nii.get_fdata()

    # Si viene con canal extra, lo eliminamos
    if vol.ndim == 4:
        vol = np.squeeze(vol)

    if vol.ndim != 3:
        raise ValueError(f"El volumen {nifti_path} no es 3D tras squeeze. Shape obtenido: {vol.shape}")

    # Mantener la misma orientación del notebook base
    vol = np.rot90(vol, k=1, axes=(0, 1))
    vol = np.flip(vol, axis=0)

    return nii, vol

# Extraer índice numérico del nombre del PNG
def extraer_indice_slice(nombre_archivo):
    nums = re.findall(r'\d+', nombre_archivo.stem)
    if not nums:
        return None
    return int(nums[-1])


# Cargar máscaras PNG de una carpeta
def load_mask(mask_path):
    """Lee una máscara PNG blanco y negro → binaria (0/1)."""
    img = Image.open(mask_path).convert("L")
    mask = np.array(img, dtype=np.uint8)
    mask = (mask > 127).astype(np.uint8)
    return mask
    
def cargar_mascaras_png(mask_dir, vol_shape=None):
    """
    Carga máscaras PNG y devuelve diccionario {slice_idx: máscara_binaria}.
    Se asume patrón de nombre tipo: '12-cerebro.png' o similar.
    """
    mascaras = {}

    if not mask_dir.exists():
        return mascaras

    for png_path in sorted(mask_dir.glob("*.png")):
        fname = png_path.name

        try:
            slice_idx = int(fname.split("-")[0]) - 1
        except Exception:
            continue

        if vol_shape is not None:
            if slice_idx < 0 or slice_idx >= vol_shape[2]:
                continue

        mask = load_mask(png_path)
        mascaras[slice_idx] = mask

    return mascaras


# Buscar casos y cargar datos
dataset_path = Path(DATASET_DIR)
case_dirs = sorted(
    [p for p in dataset_path.iterdir() if p.is_dir()]
)

casos_validos = []

for case_dir in case_dirs:
    nifti_path = case_dir / "Seq No.nii"
    cerebro_dir = case_dir / "cerebro"
    isquemia_dir = case_dir / "isquemia"

    if not nifti_path.exists():
        print(f"[AVISO] No existe: {nifti_path}")
        continue

    try:
        nii, vol = cargar_volumen_nifti(nifti_path)
        total_slices = vol.shape[2]

        # Descartar primeras y últimas 3 slices
        slice_inicio = 3
        slice_fin = total_slices - 4

        if slice_fin < slice_inicio:
            print(f"[AVISO] {case_dir.name}: no tiene suficientes slices tras descartar bordes.")
            continue

        slice_media = (slice_inicio + slice_fin) // 2

        mascaras_cerebro = cargar_mascaras_png(cerebro_dir, vol.shape)
        mascaras_isquemia = cargar_mascaras_png(isquemia_dir, vol.shape)

        casos_validos.append({
            "case": case_dir.name,
            "nii": nii,
            "vol": vol,
            "nifti_path": nifti_path,
            "slice_inicio": slice_inicio,
            "slice_fin": slice_fin,
            "slice_media": slice_media,
            "mascaras_cerebro": mascaras_cerebro,
            "mascaras_isquemia": mascaras_isquemia
        })

    except Exception as e:
        print(f"[ERROR] {case_dir.name}: {e}")

print(f"\nCasos válidos encontrados: {len(casos_validos)}")

# Mostrar slice intermedia de cada caso
for caso in casos_validos:
    vol = caso["vol"]
    s = caso["slice_media"]

    img_slice = vol[:, :, s]

    mask_cerebro = caso["mascaras_cerebro"].get(s, None)
    mask_isquemia = caso["mascaras_isquemia"].get(s, None)

    if mask_cerebro is None:
        mask_cerebro = np.zeros_like(img_slice, dtype=np.uint8)

    if mask_isquemia is None:
        mask_isquemia = np.zeros_like(img_slice, dtype=np.uint8)

    # Si la máscara no coincide en tamaño con la MRI, la redimensionamos
    if mask_cerebro.shape != img_slice.shape:
        mask_cerebro = np.array(
            Image.fromarray(mask_cerebro.astype(np.uint8)).resize(
                (img_slice.shape[1], img_slice.shape[0]),
                resample=Image.NEAREST
            ),
            dtype=np.uint8
        )
        mask_cerebro = (mask_cerebro > 0).astype(np.uint8)

    if mask_isquemia.shape != img_slice.shape:
        mask_isquemia = np.array(
            Image.fromarray(mask_isquemia.astype(np.uint8)).resize(
                (img_slice.shape[1], img_slice.shape[0]),
                resample=Image.NEAREST
            ),
            dtype=np.uint8
        )
        mask_isquemia = (mask_isquemia > 0).astype(np.uint8)

    plt.figure(figsize=(18, 4))

    # Imagen
    plt.subplot(1, 5, 1)
    plt.imshow(img_slice, cmap="gray")
    plt.title(f'{caso["case"]} - MRI slice {s}')
    plt.axis("off")

    # Máscara cerebro
    plt.subplot(1, 5, 2)
    plt.imshow(mask_cerebro, cmap="gray")
    plt.title("Máscara cerebro")
    plt.axis("off")

    # Máscara isquemia
    plt.subplot(1, 5, 3)
    plt.imshow(mask_isquemia, cmap="gray")
    plt.title("Máscara isquemia")
    plt.axis("off")

    # Superposición cerebro
    plt.subplot(1, 5, 4)
    plt.imshow(img_slice, cmap="gray")
    plt.imshow(mask_cerebro, cmap="Greens", alpha=0.4)
    plt.title("Overlay cerebro")
    plt.axis("off")

    # Superposición isquemia
    plt.subplot(1, 5, 5)
    plt.imshow(img_slice, cmap="gray")
    plt.imshow(mask_isquemia, cmap="Reds", alpha=0.4)
    plt.title("Overlay isquemia")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Carga y segmentación automática del dateset de testeo con sus respectivos modelos

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
from keras.models import load_model
import tensorflow as tf

# --- CONFIG ---
DATASET_DIR = "test_dataset"
MODEL_BRAIN = "models/brain_unet_model.h5"
MODEL_ISQ = "models/isquemia_unet_model.h5"
# Resolución de entrada empleada por los modelos U-Net
IMG_TARGET = 120 
#Tresholds
BRAIN_THRESHOLD = 0.5
ISQ_THRESHOLD = 0.8

# Funciones personalizadas
def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

def obtener_tamano_voxel(nii):
    return nii.header.get_zooms()[:3]

def load_mask(mask_path):
    """Lee una máscara PNG blanco y negro → binaria (0/1)."""
    img = Image.open(mask_path).convert("L")
    mask = np.array(img, dtype=np.uint8)
    mask = (mask > 127).astype(np.uint8)
    return mask

def cargar_volumen_nifti(nifti_path):
    nii = nib.load(str(nifti_path))
    vol = nii.get_fdata()

    if vol.ndim == 4:
        vol = np.squeeze(vol)

    if vol.ndim != 3:
        raise ValueError(f"El volumen {nifti_path} no es 3D tras squeeze. Shape: {vol.shape}")

    vol = (vol - np.min(vol)) / (np.max(vol) - np.min(vol) + 1e-8)
    vol = np.rot90(vol, k=1, axes=(0, 1))
    vol = np.flip(vol, axis=0)

    return nii, vol

def cargar_mascaras_png(mask_dir, vol_shape=None):
    """
    Devuelve diccionario {slice_idx: mask_path}
    Se asume patrón tipo '12-algo.png' -> slice_idx = 11
    """
    mascaras = {}

    if not mask_dir.exists():
        return mascaras

    for png_path in sorted(mask_dir.glob("*.png")):
        fname = png_path.name
        try:
            slice_idx = int(fname.split("-")[0]) - 1
        except Exception:
            continue

        if vol_shape is not None and (slice_idx < 0 or slice_idx >= vol_shape[2]):
            continue

        mascaras[slice_idx] = png_path

    return mascaras

def resize_img_slice(slice_img):
    """Redimensiona MRI a 120x120 si hace falta."""
    if slice_img.shape != (IMG_TARGET, IMG_TARGET):
        return np.array(
            Image.fromarray((slice_img * 255).astype(np.uint8)).resize((IMG_TARGET, IMG_TARGET))
        ).astype(np.float32) / 255.0
    return slice_img.astype(np.float32)

def resize_mask(mask):
    """Redimensiona máscara a 120x120 con nearest neighbor."""
    if mask.shape != (IMG_TARGET, IMG_TARGET):
        mask = np.array(
            Image.fromarray(mask.astype(np.uint8)).resize(
                (IMG_TARGET, IMG_TARGET),
                resample=Image.NEAREST
            ),
            dtype=np.uint8
        )
    return (mask > 0).astype(np.uint8)

def dice_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)

    inter = np.sum(y_true * y_pred)
    total = np.sum(y_true) + np.sum(y_pred)

    if total == 0:
        return 1.0

    return (2.0 * inter + smooth) / (total + smooth)

def iou_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)

    inter = np.sum(y_true * y_pred)
    union = np.sum((y_true + y_pred) > 0)

    if union == 0:
        return 1.0

    return (inter + smooth) / (union + smooth)

def volumen_ml(mask_3d, voxel_size):
    voxel_volume_mm3 = voxel_size[0] * voxel_size[1] * voxel_size[2]
    voxels = np.sum(mask_3d)
    return (voxels * voxel_volume_mm3) / 1000.0


# Cargar modelos
model_brain = load_model(MODEL_BRAIN, custom_objects={
    "dice_coef": dice_coef,
    "dice_loss": dice_loss,
    "bce_dice_loss": bce_dice_loss
})

model_isq = load_model(MODEL_ISQ, custom_objects={
    "dice_coef": dice_coef,
    "dice_loss": dice_loss,
    "bce_dice_loss": bce_dice_loss
})

print("Modelos cargados correctamente")

# Recorrido del dataset
dataset_path = Path(DATASET_DIR)

case_dirs = sorted(
    [p for p in dataset_path.iterdir() if p.is_dir()],
    key=lambda x: x.name.lower()
)

resultados_casos = []
resultados_slices = []

for case_dir in case_dirs:
    nifti_path = case_dir / "Seq No.nii"
    cerebro_dir = case_dir / "cerebro"
    isquemia_dir = case_dir / "isquemia"

    if not nifti_path.exists():
        print(f"❌ {case_dir.name}: no existe Seq No.nii")
        continue

    try:
        nii, vol = cargar_volumen_nifti(nifti_path)
        voxel_size = obtener_tamano_voxel(nii)

        total_slices = vol.shape[2]
        slice_inicio = 3
        slice_fin = total_slices - 4

        if slice_fin < slice_inicio:
            print(f"❌ {case_dir.name}: no tiene suficientes slices tras descartar 3 al inicio y 3 al final")
            continue

        masks_brain_paths = cargar_mascaras_png(cerebro_dir, vol.shape)
        masks_isq_paths = cargar_mascaras_png(isquemia_dir, vol.shape)

        # Volúmenes 3D completos, ya en resolución correcta
        num_slices_validas = slice_fin - slice_inicio + 1
        mask_brain_pred_3d = np.zeros((IMG_TARGET, IMG_TARGET, num_slices_validas), dtype=np.uint8)
        mask_isq_pred_3d   = np.zeros((IMG_TARGET, IMG_TARGET, num_slices_validas), dtype=np.uint8)
        mask_brain_real_3d = np.zeros((IMG_TARGET, IMG_TARGET, num_slices_validas), dtype=np.uint8)
        mask_isq_real_3d   = np.zeros((IMG_TARGET, IMG_TARGET, num_slices_validas), dtype=np.uint8)

        for out_idx, i in enumerate(range(slice_inicio, slice_fin + 1)):
            slice_img = np.squeeze(vol[:, :, i])
            slice_img_resized = resize_img_slice(slice_img)

            input_img = slice_img_resized[np.newaxis, :, :, np.newaxis].astype(np.float32)

            # Predicción cerebro
            mask_brain = model_brain.predict(input_img, verbose=0)[0, ..., 0]
            mask_brain_bin = (mask_brain > BRAIN_THRESHOLD).astype(np.uint8)

        
            # Multiplicación capa isquemia
            input_img_brain = input_img * mask_brain_bin[np.newaxis, :, :, np.newaxis]

            # Predicción isquemia
            mask_isq = model_isq.predict(input_img_brain, verbose=0)[0, ..., 0]
            mask_isq_bin = (mask_isq > ISQ_THRESHOLD).astype(np.uint8)

            # Máscaras reales
            if i in masks_brain_paths:
                mask_brain_real = resize_mask(load_mask(masks_brain_paths[i]))
            else:
                mask_brain_real = np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.uint8)

            if i in masks_isq_paths:
                mask_isq_real = resize_mask(load_mask(masks_isq_paths[i]))
            else:
                mask_isq_real = np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.uint8)

            # Guardar en 3D
            mask_brain_pred_3d[:, :, out_idx] = mask_brain_bin
            mask_isq_pred_3d[:, :, out_idx]   = mask_isq_bin
            mask_brain_real_3d[:, :, out_idx] = mask_brain_real
            mask_isq_real_3d[:, :, out_idx]   = mask_isq_real

            # Métricas por slice
            brain_pred_px = int(np.sum(mask_brain_bin))
            brain_real_px = int(np.sum(mask_brain_real))
            isq_pred_px   = int(np.sum(mask_isq_bin))
            isq_real_px   = int(np.sum(mask_isq_real))

            resultados_slices.append({
                "case": case_dir.name,
                "slice_idx_original": i,
                "slice_idx_analisis": out_idx,
                "brain_pred_px": brain_pred_px,
                "brain_real_px": brain_real_px,
                "brain_diff_px": brain_pred_px - brain_real_px,
                "brain_abs_diff_px": abs(brain_pred_px - brain_real_px),
                "brain_dice": dice_numpy(mask_brain_real, mask_brain_bin),
                "brain_iou": iou_numpy(mask_brain_real, mask_brain_bin),
                "isq_pred_px": isq_pred_px,
                "isq_real_px": isq_real_px,
                "isq_diff_px": isq_pred_px - isq_real_px,
                "isq_abs_diff_px": abs(isq_pred_px - isq_real_px),
                "isq_dice": dice_numpy(mask_isq_real, mask_isq_bin),
                "isq_iou": iou_numpy(mask_isq_real, mask_isq_bin),
            })

        # Métricas por caso (3D)
        brain_pred_vol_ml = volumen_ml(mask_brain_pred_3d, voxel_size)
        brain_real_vol_ml = volumen_ml(mask_brain_real_3d, voxel_size)
        isq_pred_vol_ml   = volumen_ml(mask_isq_pred_3d, voxel_size)
        isq_real_vol_ml   = volumen_ml(mask_isq_real_3d, voxel_size)

        porcentaje_pred = (isq_pred_vol_ml / brain_pred_vol_ml * 100) if brain_pred_vol_ml > 0 else 0.0
        porcentaje_real = (isq_real_vol_ml / brain_real_vol_ml * 100) if brain_real_vol_ml > 0 else 0.0

        resultados_casos.append({
            "case": case_dir.name,
            "n_slices_total": total_slices,
            "slice_inicio": slice_inicio,
            "slice_fin": slice_fin,
            "n_slices_analizadas": num_slices_validas,

            "voxel_x_mm": voxel_size[0],
            "voxel_y_mm": voxel_size[1],
            "voxel_z_mm": voxel_size[2],

            "brain_pred_vol_ml": brain_pred_vol_ml,
            "brain_real_vol_ml": brain_real_vol_ml,
            "brain_diff_vol_ml": brain_pred_vol_ml - brain_real_vol_ml,
            "brain_abs_diff_vol_ml": abs(brain_pred_vol_ml - brain_real_vol_ml),
            "brain_rel_error_pct": (
                abs(brain_pred_vol_ml - brain_real_vol_ml) / brain_real_vol_ml * 100
                if brain_real_vol_ml > 0 else np.nan
            ),
            "brain_dice_3d": dice_numpy(mask_brain_real_3d, mask_brain_pred_3d),
            "brain_iou_3d": iou_numpy(mask_brain_real_3d, mask_brain_pred_3d),

            "isq_pred_vol_ml": isq_pred_vol_ml,
            "isq_real_vol_ml": isq_real_vol_ml,
            "isq_diff_vol_ml": isq_pred_vol_ml - isq_real_vol_ml, 
            "isq_abs_diff_vol_ml": abs(isq_pred_vol_ml - isq_real_vol_ml),
            "isq_rel_error_pct": (
                abs(isq_pred_vol_ml - isq_real_vol_ml) / isq_real_vol_ml * 100
                if isq_real_vol_ml > 0 else np.nan
            ),
            "isq_dice_3d": dice_numpy(mask_isq_real_3d, mask_isq_pred_3d),
            "isq_iou_3d": iou_numpy(mask_isq_real_3d, mask_isq_pred_3d),

            "afectacion_pred_pct": porcentaje_pred,
            "afectacion_real_pct": porcentaje_real,
            "afectacion_diff_pct": porcentaje_pred - porcentaje_real,
            "afectacion_abs_diff_pct": abs(porcentaje_pred - porcentaje_real),
        })

        print(f"✅ {case_dir.name} procesado")

    except Exception as e:
        print(f"❌ Error en {case_dir.name}: {e}")

# DataFrames finales
df_casos = pd.DataFrame(resultados_casos).sort_values("case").reset_index(drop=True)
df_slices = pd.DataFrame(resultados_slices).sort_values(["case", "slice_idx_original"]).reset_index(drop=True)

print("\n=============================")
print("RESUMEN DEL ANÁLISIS")
print("=============================")
print(f"Casos procesados: {len(df_casos)}")
print(f"Slices analizadas: {len(df_slices)}")
print(f"THRESHOLD (Isquemia): {ISQ_THRESHOLD }")

if len(df_casos) > 0:
    print("\nColumnas disponibles en df_casos:")
    print(df_casos.columns.tolist())

if len(df_slices) > 0:
    print("\nColumnas disponibles en df_slices:")
    print(df_slices.columns.tolist())

display(df_casos.head())
display(df_slices.head())

# Gráficas

## Volumen segmentación manual vs volumen predicho

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Preparar datos
df_plot = df_casos.copy()

# Asegurar orden estable
df_plot = df_plot.sort_values("case").reset_index(drop=True)

# Recalcular volúmenes de ISQUEMIA desde píxeles de isquemia
if {"case", "isq_real_px", "isq_pred_px"}.issubset(df_slices.columns):

    voxel_info = df_plot[[
        "case",
        "voxel_x_mm",
        "voxel_y_mm",
        "voxel_z_mm"
    ]].copy()

    voxel_info["voxel_volume_ml"] = (
        voxel_info["voxel_x_mm"] *
        voxel_info["voxel_y_mm"] *
        voxel_info["voxel_z_mm"]
    ) / 1000.0

    isq_px_por_caso = (
        df_slices
        .groupby("case", as_index=False)
        .agg(
            isq_real_px_total=("isq_real_px", "sum"),
            isq_pred_px_total=("isq_pred_px", "sum")
        )
    )

    isq_vol_recalc = isq_px_por_caso.merge(
        voxel_info[["case", "voxel_volume_ml"]],
        on="case",
        how="left"
    )

    isq_vol_recalc["isq_real_vol_ml_recalc"] = (
        isq_vol_recalc["isq_real_px_total"] *
        isq_vol_recalc["voxel_volume_ml"]
    )

    isq_vol_recalc["isq_pred_vol_ml_recalc"] = (
        isq_vol_recalc["isq_pred_px_total"] *
        isq_vol_recalc["voxel_volume_ml"]
    )

    df_plot = df_plot.merge(
        isq_vol_recalc[[
            "case",
            "isq_real_vol_ml_recalc",
            "isq_pred_vol_ml_recalc"
        ]],
        on="case",
        how="left"
    )

    df_plot["isq_real_vol_ml"] = df_plot["isq_real_vol_ml_recalc"]
    df_plot["isq_pred_vol_ml"] = df_plot["isq_pred_vol_ml_recalc"]

    df_plot = df_plot.drop(
        columns=[
            "isq_real_vol_ml_recalc",
            "isq_pred_vol_ml_recalc"
        ]
    )

else:
    print("[AVISO] No se han encontrado columnas isq_real_px / isq_pred_px en df_slices.")
    print("[AVISO] Se usan los volúmenes de isquemia ya existentes en df_casos.")


# Métricas visuales

# Cerebro
df_plot["brain_diff_ml_signed"] = (
    df_plot["brain_pred_vol_ml"] -
    df_plot["brain_real_vol_ml"]
)

df_plot["brain_diff_ml_abs"] = np.abs(df_plot["brain_diff_ml_signed"])

df_plot["brain_error_pct"] = np.where(
    df_plot["brain_real_vol_ml"] > 0,
    df_plot["brain_diff_ml_abs"] / df_plot["brain_real_vol_ml"] * 100,
    np.nan
)

df_plot["brain_accuracy_pct"] = (
    100 - df_plot["brain_error_pct"]
).clip(lower=0, upper=100)


# Isquemia
df_plot["isq_diff_ml_signed"] = (
    df_plot["isq_pred_vol_ml"] -
    df_plot["isq_real_vol_ml"]
)

df_plot["isq_diff_ml_abs"] = np.abs(df_plot["isq_diff_ml_signed"])

df_plot["isq_error_pct"] = np.where(
    df_plot["isq_real_vol_ml"] > 0,
    df_plot["isq_diff_ml_abs"] / df_plot["isq_real_vol_ml"] * 100,
    np.nan
)

df_plot["isq_accuracy_pct"] = (
    100 - df_plot["isq_error_pct"]
).clip(lower=0, upper=100)


cases = df_plot["case"].astype(str)
x = np.arange(len(cases))
width = 0.35


# Función para escribir valores en barras
def poner_valores_barras(ax, bars, decimales=4):
    for bar in bars:
        altura = bar.get_height()

        if np.isnan(altura):
            continue

        ax.annotate(
            f"{altura:.{decimales}f}",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                altura
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )


# CEREBRO - Volumen manual vs IA
fig, ax = plt.subplots(figsize=(15, 6))

bars1 = ax.bar(
    x - width/2,
    df_plot["brain_real_vol_ml"],
    width,
    label="Cerebro Real"
)

bars2 = ax.bar(
    x + width/2,
    df_plot["brain_pred_vol_ml"],
    width,
    label="Cerebro IA"
)

poner_valores_barras(ax, bars1)
poner_valores_barras(ax, bars2)

ax.set_xticks(x)
ax.set_xticklabels(cases, rotation=45, ha="right")
ax.set_ylabel("Volumen (ml)")
ax.set_title("Comparación de Volumen Cerebral: Real vs IA")
ax.legend()

ax.set_ylim(
    0,
    max(
        df_plot["brain_real_vol_ml"].max(),
        df_plot["brain_pred_vol_ml"].max()
    ) * 1.18
)

plt.tight_layout()
plt.show()


# CEREBRO - Diferencia absoluta
fig, ax = plt.subplots(figsize=(15, 5))

bars = ax.bar(x, df_plot["brain_diff_ml_abs"])

poner_valores_barras(ax, bars)

ax.set_xticks(x)
ax.set_xticklabels(cases, rotation=45, ha="right")
ax.set_ylabel("Diferencia absoluta (ml)")
ax.set_title("Error Absoluto en Volumen Cerebral")

ax.set_ylim(
    0,
    df_plot["brain_diff_ml_abs"].max() * 1.18
    if df_plot["brain_diff_ml_abs"].max() > 0 else 1
)

plt.tight_layout()
plt.show()


# CEREBRO - Error % y acierto %
plt.figure(figsize=(15, 5))

plt.plot(
    x,
    df_plot["brain_error_pct"],
    marker="o",
    label="Error relativo (%)"
)

plt.plot(
    x,
    df_plot["brain_accuracy_pct"],
    marker="s",
    label="Acierto (%)"
)

plt.xticks(x, cases, rotation=45, ha="right")
plt.ylabel("Porcentaje (%)")
plt.title("Fiabilidad del Volumen Cerebral Predicho")
plt.legend()
plt.tight_layout()
plt.show()


# ISQUEMIA - Volumen manual vs IA
fig, ax = plt.subplots(figsize=(15, 6))

bars1 = ax.bar(
    x - width/2,
    df_plot["isq_real_vol_ml"],
    width,
    label="Isquemia Real"
)

bars2 = ax.bar(
    x + width/2,
    df_plot["isq_pred_vol_ml"],
    width,
    label="Isquemia IA"
)

poner_valores_barras(ax, bars1)
poner_valores_barras(ax, bars2)

ax.set_xticks(x)
ax.set_xticklabels(cases, rotation=45, ha="right")
ax.set_ylabel("Volumen (ml)")
ax.set_title("Comparación de Volumen de Isquemia: Real vs IA")
ax.legend()

max_isq = max(
    df_plot["isq_real_vol_ml"].max(),
    df_plot["isq_pred_vol_ml"].max()
)

ax.set_ylim(0, max_isq * 1.25 if max_isq > 0 else 1)

plt.tight_layout()
plt.show()



# ISQUEMIA - Diferencia absoluta
fig, ax = plt.subplots(figsize=(15, 5))

bars = ax.bar(x, df_plot["isq_diff_ml_abs"])

poner_valores_barras(ax, bars)

ax.set_xticks(x)
ax.set_xticklabels(cases, rotation=45, ha="right")
ax.set_ylabel("Diferencia absoluta (ml)")
ax.set_title("Error Absoluto en Volumen de Isquemia")

ax.set_ylim(
    0,
    df_plot["isq_diff_ml_abs"].max() * 1.25
    if df_plot["isq_diff_ml_abs"].max() > 0 else 1
)

plt.tight_layout()
plt.show()


# ISQUEMIA - Error % y acierto %
plt.figure(figsize=(15, 5))

plt.plot(
    x,
    df_plot["isq_error_pct"],
    marker="o",
    label="Error relativo (%)"
)

plt.plot(
    x,
    df_plot["isq_accuracy_pct"],
    marker="s",
    label="Acierto (%)"
)

plt.xticks(x, cases, rotation=45, ha="right")
plt.ylabel("Porcentaje (%)")
plt.title("Fiabilidad del Volumen de Isquemia Predicho")
plt.legend()
plt.tight_layout()
plt.show()


# Resumen numérico global
resumen = pd.DataFrame({
    "Métrica": [
        "Error absoluto medio cerebro (ml)",
        "Error relativo medio cerebro (%)",
        "Acierto medio cerebro (%)",
        "Error absoluto medio isquemia (ml)",
        "Error relativo medio isquemia (%)",
        "Acierto medio isquemia (%)"
    ],
    "Valor": [
        df_plot["brain_diff_ml_abs"].mean(),
        df_plot["brain_error_pct"].mean(),
        df_plot["brain_accuracy_pct"].mean(),
        df_plot["isq_diff_ml_abs"].mean(),
        df_plot["isq_error_pct"].mean(),
        df_plot["isq_accuracy_pct"].mean()
    ]
})

print("===== RESUMEN GLOBAL DE FIABILIDAD =====")
display(resumen.round(4))


print("===== COMPROBACIÓN DE VOLÚMENES DE ISQUEMIA =====")
display(
    df_plot[[
        "case",
        "isq_real_vol_ml",
        "isq_pred_vol_ml",
        "brain_real_vol_ml",
        "brain_pred_vol_ml"
    ]].round(4)
)

## Scatter plot (volumen real vs predicho)

### Cómo interpretar esta gráfica:

La línea representa predicción perfecta.
Cuanto más cerca estén los puntos de la línea, mejor es el modelo.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# CEREBRO
plt.figure(figsize=(7,7))

real = df_casos["brain_real_vol_ml"]
pred = df_casos["brain_pred_vol_ml"]

plt.scatter(real, pred)

# línea ideal
min_v = min(real.min(), pred.min())
max_v = max(real.max(), pred.max())
plt.plot([min_v, max_v], [min_v, max_v])

plt.xlabel("Volumen real cerebro (ml)")
plt.ylabel("Volumen predicho IA (ml)")
plt.title("Volumen cerebral: Real vs IA")

plt.tight_layout()
plt.show()

# ISQUEMIA
plt.figure(figsize=(7,7))

real = df_casos["isq_real_vol_ml"]
pred = df_casos["isq_pred_vol_ml"]

plt.scatter(real, pred)

min_v = min(real.min(), pred.min())
max_v = max(real.max(), pred.max())
plt.plot([min_v, max_v], [min_v, max_v])

plt.xlabel("Volumen real isquemia (ml)")
plt.ylabel("Volumen predicho IA (ml)")
plt.title("Volumen de isquemia: Real vs IA")

plt.tight_layout()
plt.show()

## Dice score por paciente

### Interpretación del Dice Score

El **Dice score** es una métrica ampliamente utilizada en tareas de **segmentación médica**, ya que mide el grado de solapamiento entre la segmentación realizada por el modelo y la segmentación real (ground truth).

Su definición es:

\[
Dice = \frac{2TP}{2TP + FP + FN}
\]

donde:

- **TP (True Positives)**: píxeles correctamente segmentados.
- **FP (False Positives)**: píxeles segmentados por el modelo pero que no pertenecen a la región real.
- **FN (False Negatives)**: píxeles reales que el modelo no detectó.

El valor del Dice score varía entre **0 y 1**:

| Dice Score | Interpretación |
|-------------|----------------|
| **> 0.9** | Segmentación excelente |
| **0.8 – 0.9** | Segmentación muy buena |
| **0.7 – 0.8** | Segmentación buena |
| **0.6 – 0.7** | Segmentación aceptable |
| **< 0.6** | Segmentación pobre |

Un Dice cercano a **1** indica que la máscara predicha por el modelo coincide casi perfectamente con la máscara real.

En este análisis, el Dice score se calcula **para cada paciente**, lo que permite evaluar:

- La **consistencia del modelo** en distintos casos clínicos.
- La presencia de **casos especialmente difíciles**.
- La **variabilidad del rendimiento del modelo** en el conjunto de datos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cases = df_casos["case"]
x = np.arange(len(cases))

# Cerebro
plt.figure(figsize=(14,5))

plt.plot(
    x,
    df_casos["brain_dice_3d"],
    marker="o",
    label="Dice cerebro"
)

plt.xticks(x, cases, rotation=45)
plt.ylabel("Dice score")
plt.ylim(0,1)
plt.title("Dice score por paciente - Segmentación de cerebro")

plt.legend()
plt.tight_layout()
plt.show()

# Isquemia
plt.figure(figsize=(14,5))

plt.plot(
    x,
    df_casos["isq_dice_3d"],
    marker="o",
    label="Dice isquemia"
)

plt.xticks(x, cases, rotation=45)
plt.ylabel("Dice score")
plt.ylim(0,1)
plt.title("Dice score por paciente - Segmentación de isquemia")

plt.legend()
plt.tight_layout()
plt.show()

# Media y desviación estándar
print("===== RESUMEN DICE =====")

print("\nCerebro")
print("Dice medio:", round(df_casos["brain_dice_3d"].mean(),3))
print("Desviación estándar:", round(df_casos["brain_dice_3d"].std(),3))

print("\nIsquemia")
print("Dice medio:", round(df_casos["isq_dice_3d"].mean(),3))
print("Desviación estándar:", round(df_casos["isq_dice_3d"].std(),3))

## Análisis Bland-Altman

El **Bland-Altman plot** es una herramienta estadística utilizada para evaluar la concordancia entre dos métodos de medición. En este caso se emplea para comparar los volúmenes obtenidos mediante segmentación manual (ground truth) y los generados por el modelo de inteligencia artificial.

En este gráfico:

- El **eje X** representa la media entre el volumen real y el volumen predicho.
- El **eje Y** representa la diferencia entre ambos valores (IA − Real).

Se incluyen además tres líneas de referencia:

- **Sesgo medio**: representa la diferencia promedio entre ambos métodos.
- **Límites de concordancia**: definidos como la media ± 1.96 desviaciones estándar, que contienen aproximadamente el 95 % de las diferencias.

La interpretación del gráfico permite identificar:

- Si el modelo presenta un **sesgo sistemático**, es decir, si tiende a sobreestimar o infraestimar los volúmenes.
- La **variabilidad del error** entre casos.
- Si el error depende del **tamaño de la estructura segmentada**, lo cual es común en lesiones pequeñas.

Una distribución de puntos centrada alrededor de la línea de sesgo y dentro de los límites de concordancia indica una buena concordancia entre el método automático y la segmentación manual.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# CEREBRO
real = df_casos["brain_real_vol_ml"]
pred = df_casos["brain_pred_vol_ml"]

mean_vals = (real + pred) / 2
diff_vals = pred - real

mean_diff = np.mean(diff_vals)
std_diff = np.std(diff_vals)

plt.figure(figsize=(7,7))

plt.scatter(mean_vals, diff_vals)

plt.axhline(mean_diff, linestyle="--", label="Sesgo medio")
plt.axhline(mean_diff + 1.96*std_diff, linestyle=":")
plt.axhline(mean_diff - 1.96*std_diff, linestyle=":")

plt.xlabel("Media de volumen (ml)")
plt.ylabel("Diferencia IA - Real (ml)")
plt.title("Bland-Altman: Segmentación de cerebro")

plt.legend()
plt.tight_layout()
plt.show()


# ISQUEMIA
real = df_casos["isq_real_vol_ml"]
pred = df_casos["isq_pred_vol_ml"]

mean_vals = (real + pred) / 2
diff_vals = pred - real

mean_diff = np.mean(diff_vals)
std_diff = np.std(diff_vals)

plt.figure(figsize=(7,7))

plt.scatter(mean_vals, diff_vals)

plt.axhline(mean_diff, linestyle="--", label="Sesgo medio")
plt.axhline(mean_diff + 1.96*std_diff, linestyle=":")
plt.axhline(mean_diff - 1.96*std_diff, linestyle=":")

plt.xlabel("Media de volumen (ml)")
plt.ylabel("Diferencia IA - Real (ml)")
plt.title("Bland-Altman: Segmentación de isquemia")

plt.legend()
plt.tight_layout()
plt.show()

## Distribución del rendimiento del modelo

Para analizar la variabilidad del rendimiento del modelo se emplean diagramas de caja (*boxplots*), los cuales permiten visualizar la distribución de diferentes métricas de evaluación.

En este caso se representan dos métricas principales:

- **Dice score**, que mide el grado de solapamiento entre la segmentación predicha y la segmentación manual.
- **Error volumétrico relativo**, que indica la diferencia porcentual entre el volumen real y el volumen estimado por el modelo.

Cada boxplot muestra:

- **La mediana** de la distribución (línea central).
- **El rango intercuartílico (IQR)**, que contiene el 50 % central de los valores.
- **Los valores extremos u outliers**, que representan casos en los que el modelo presenta un comportamiento diferente al resto.

Este tipo de representación permite evaluar de forma visual:

- La **estabilidad del modelo** en distintos pacientes.
- La **variabilidad del rendimiento** entre casos.
- La presencia de **posibles outliers**, que pueden corresponder a casos clínicos más complejos o a lesiones de menor tamaño.

In [ ]:
import matplotlib.pyplot as plt

# BOXPLOT DICE
plt.figure(figsize=(6,6))

data = [
    df_casos["brain_dice_3d"],
    df_casos["isq_dice_3d"]
]

plt.boxplot(data)

plt.xticks([1,2], ["Cerebro", "Isquemia"])
plt.ylabel("Dice score")
plt.title("Distribución del Dice score")

plt.tight_layout()
plt.show()


# BOXPLOT ERROR RELATIVO
plt.figure(figsize=(6,6))

data = [
    df_casos["brain_rel_error_pct"],
    df_casos["isq_rel_error_pct"]
]

plt.boxplot(data)

plt.xticks([1,2], ["Cerebro", "Isquemia"])
plt.ylabel("Error relativo (%)")
plt.title("Distribución del error volumétrico")

plt.tight_layout()
plt.show()

## Distribución del rendimiento del modelo

Para analizar cómo se distribuye el rendimiento del modelo en el conjunto de datos se utilizan histogramas de las métricas principales de evaluación.

En este análisis se representan:

- **Dice score**, que mide el grado de solapamiento entre la segmentación predicha y la segmentación manual.
- **Error volumétrico relativo**, que mide la diferencia porcentual entre el volumen real y el volumen estimado por el modelo.

Los histogramas permiten observar:

- La **concentración de resultados** en determinados rangos de rendimiento.
- La presencia de **casos extremos o atípicos**.
- La **estabilidad global del modelo** en el conjunto de pacientes.

Una distribución concentrada en valores altos de Dice y en valores bajos de error volumétrico indica que el modelo presenta un comportamiento consistente y fiable en la mayoría de los casos analizados.

In [ ]:
import matplotlib.pyplot as plt

# HISTOGRAMA DICE CEREBRO
plt.figure(figsize=(6,5))

plt.hist(df_casos["brain_dice_3d"], bins=10)

plt.xlabel("Dice score")
plt.ylabel("Número de casos")
plt.title("Distribución del Dice score - Cerebro")

plt.tight_layout()
plt.show()


# HISTOGRAMA DICE ISQUEMIA
plt.figure(figsize=(6,5))

plt.hist(df_casos["isq_dice_3d"], bins=10)

plt.xlabel("Dice score")
plt.ylabel("Número de casos")
plt.title("Distribución del Dice score - Isquemia")

plt.tight_layout()
plt.show()


# HISTOGRAMA ERROR CEREBRO
plt.figure(figsize=(6,5))

plt.hist(df_casos["brain_rel_error_pct"], bins=10)

plt.xlabel("Error relativo (%)")
plt.ylabel("Número de casos")
plt.title("Distribución del error volumétrico - Cerebro")

plt.tight_layout()
plt.show()


# HISTOGRAMA ERROR ISQUEMIA
plt.figure(figsize=(6,5))

plt.hist(df_casos["isq_rel_error_pct"], bins=10)

plt.xlabel("Error relativo (%)")
plt.ylabel("Número de casos")
plt.title("Distribución del error volumétrico - Isquemia")

plt.tight_layout()
plt.show()

# Comparación visual de segmentaciones

Además del análisis cuantitativo mediante métricas estadísticas, resulta fundamental realizar una evaluación cualitativa del modelo mediante la inspección visual de las segmentaciones obtenidas.

En esta sección se muestran ejemplos representativos de diferentes pacientes del conjunto de datos, incluyendo:

- **Imagen MRI original**
- **Segmentación manual (ground truth)**
- **Segmentación generada por el modelo**
- **Mapa de diferencias entre ambas segmentaciones**

Las máscaras se representan sobre la imagen original utilizando colores distintos:

- **Verde**: segmentación cerebral
- **Rojo**: región de isquemia

El mapa de diferencias permite identificar visualmente las regiones donde la segmentación automática difiere de la segmentación manual.

Este tipo de representación cualitativa permite evaluar aspectos que no siempre se reflejan completamente en las métricas numéricas, como la precisión en los bordes de la lesión, la presencia de falsos positivos o la omisión de regiones pequeñas de tejido afectado.

In [ ]:
# RESUMEN FINAL DEL MODELO + INFORME EXCEL
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

# 1) MÉTRICAS GLOBALES
brain_abs_err = df_casos["brain_abs_diff_vol_ml"].mean()
brain_rel_err = df_casos["brain_rel_error_pct"].mean()
brain_acc = 100 - brain_rel_err
brain_dice = df_casos["brain_dice_3d"].mean()
brain_iou = df_casos["brain_iou_3d"].mean()

isq_abs_err = df_casos["isq_abs_diff_vol_ml"].mean()
isq_rel_err = df_casos["isq_rel_error_pct"].mean()
isq_acc = 100 - isq_rel_err
isq_dice = df_casos["isq_dice_3d"].mean()
isq_iou = df_casos["isq_iou_3d"].mean()

afect_diff = df_casos["afectacion_abs_diff_pct"].mean()

brain_abs_err_std = df_casos["brain_abs_diff_vol_ml"].std()
brain_rel_err_std = df_casos["brain_rel_error_pct"].std()
brain_dice_std = df_casos["brain_dice_3d"].std()
brain_iou_std = df_casos["brain_iou_3d"].std()

isq_abs_err_std = df_casos["isq_abs_diff_vol_ml"].std()
isq_rel_err_std = df_casos["isq_rel_error_pct"].std()
isq_dice_std = df_casos["isq_dice_3d"].std()
isq_iou_std = df_casos["isq_iou_3d"].std()

afect_diff_std = df_casos["afectacion_abs_diff_pct"].std()

# 2) CLASIFICACIÓN AUTOMÁTICA
def clasificar_segmentacion(dice, error_pct):
    if dice >= 0.90 and error_pct <= 10:
        return "excelente"
    elif dice >= 0.80 and error_pct <= 20:
        return "muy buena"
    elif dice >= 0.70 and error_pct <= 35:
        return "buena"
    elif dice >= 0.60 and error_pct <= 50:
        return "aceptable"
    else:
        return "mejorable"

calif_brain = clasificar_segmentacion(brain_dice, brain_rel_err)
calif_isq = clasificar_segmentacion(isq_dice, isq_rel_err)

if calif_brain in ["excelente", "muy buena"] and calif_isq in ["excelente", "muy buena", "buena"]:
    conclusion_global = "El modelo presenta un rendimiento global bueno y apto para análisis comparativo en el TFG."
elif calif_brain in ["excelente", "muy buena"] and calif_isq == "aceptable":
    conclusion_global = "El modelo es muy sólido en cerebro y aceptable en isquemia, aunque con mayor variabilidad en lesiones pequeñas."
else:
    conclusion_global = "El modelo ofrece resultados útiles, pero requiere mejoras para considerarse robusto en todos los casos."

# 3) RESUMEN EN PANTALLA
print("=====================================================")
print("RESUMEN FINAL DEL RENDIMIENTO DEL MODELO")
print("=====================================================")
print(f"Segmentación de cerebro:   {calif_brain.upper()}")
print(f"  - Error absoluto medio:  {brain_abs_err:.3f} ± {brain_abs_err_std:.3f} ml")
print(f"  - Error relativo medio:  {brain_rel_err:.2f} ± {brain_rel_err_std:.2f} %")
print(f"  - Acierto medio:         {brain_acc:.2f} %")
print(f"  - Dice medio 3D:         {brain_dice:.3f} ± {brain_dice_std:.3f}")
print(f"  - IoU medio 3D:          {brain_iou:.3f} ± {brain_iou_std:.3f}")
print("")
print(f"  - Error absoluto medio:  {isq_abs_err:.3f} ± {isq_abs_err_std:.3f} ml")
print(f"  - Error relativo medio:  {isq_rel_err:.2f} ± {isq_rel_err_std:.2f} %")
print(f"  - Acierto medio:         {isq_acc:.2f} %")
print(f"  - Dice medio 3D:         {isq_dice:.3f} ± {isq_dice_std:.3f}")
print(f"  - IoU medio 3D:          {isq_iou:.3f} ± {isq_iou_std:.3f}")
print("")
print(f"Diferencia media en % de afectación: {afect_diff:.3f} puntos porcentuales")
print("")
print("Conclusión global:")
print(conclusion_global)
print("=====================================================")

# 4) PREPARAR TABLAS PARA EL INFORME
df_resumen = pd.DataFrame({
    "Métrica": [
        "Clasificación cerebro",
        "Error absoluto cerebro (ml)",
        "Error relativo cerebro (%)",
        "Acierto cerebro (%)",
        "Dice cerebro",
        "IoU cerebro",

        "Clasificación isquemia",
        "Error absoluto isquemia (ml)",
        "Error relativo isquemia (%)",
        "Acierto isquemia (%)",
        "Dice isquemia",
        "IoU isquemia",

        "Diferencia % afectación",
        "Conclusión global"
    ],
    "Media": [
        calif_brain,
        brain_abs_err,
        brain_rel_err,
        brain_acc,
        brain_dice,
        brain_iou,

        calif_isq,
        isq_abs_err,
        isq_rel_err,
        isq_acc,
        isq_dice,
        isq_iou,

        afect_diff,
        conclusion_global
    ],
    "Desviación estándar": [
        "",
        brain_abs_err_std,
        brain_rel_err_std,
        "",
        brain_dice_std,
        brain_iou_std,

        "",
        isq_abs_err_std,
        isq_rel_err_std,
        "",
        isq_dice_std,
        isq_iou_std,

        afect_diff_std,
        ""
    ]
})

# 5) CARPETA DE SALIDA
output_dir = "INFORME_TFG_MODELO"
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(output_dir, "Informe_TFG_Segmentacion.xlsx")

# 6) GENERAR TODAS LAS GRÁFICAS COMO PNG
def guardar_grafica_volumenes():
    x = np.arange(len(df_casos))
    width = 0.35
    cases = df_casos["case"]

    # Cerebro
    plt.figure(figsize=(14,6))
    plt.bar(x - width/2, df_casos["brain_real_vol_ml"], width, label="Cerebro Real")
    plt.bar(x + width/2, df_casos["brain_pred_vol_ml"], width, label="Cerebro IA")
    plt.xticks(x, cases, rotation=45)
    plt.ylabel("Volumen (ml)")
    plt.title("Comparación de volumen cerebral: Real vs IA")
    plt.legend()
    plt.tight_layout()
    path1 = os.path.join(output_dir, "grafica_volumen_cerebro.png")
    plt.savefig(path1, dpi=200, bbox_inches="tight")
    plt.close()

    # Isquemia
    plt.figure(figsize=(14,6))
    plt.bar(x - width/2, df_casos["isq_real_vol_ml"], width, label="Isquemia Real")
    plt.bar(x + width/2, df_casos["isq_pred_vol_ml"], width, label="Isquemia IA")
    plt.xticks(x, cases, rotation=45)
    plt.ylabel("Volumen (ml)")
    plt.title("Comparación de volumen de isquemia: Real vs IA")
    plt.legend()
    plt.tight_layout()
    path2 = os.path.join(output_dir, "grafica_volumen_isquemia.png")
    plt.savefig(path2, dpi=200, bbox_inches="tight")
    plt.close()

    return path1, path2

def guardar_grafica_scatter():
    # Cerebro
    real = df_casos["brain_real_vol_ml"]
    pred = df_casos["brain_pred_vol_ml"]
    plt.figure(figsize=(7,7))
    plt.scatter(real, pred)
    min_v = min(real.min(), pred.min())
    max_v = max(real.max(), pred.max())
    plt.plot([min_v, max_v], [min_v, max_v])
    plt.xlabel("Volumen real cerebro (ml)")
    plt.ylabel("Volumen predicho IA (ml)")
    plt.title("Scatter plot: Cerebro")
    plt.tight_layout()
    path1 = os.path.join(output_dir, "scatter_cerebro.png")
    plt.savefig(path1, dpi=200, bbox_inches="tight")
    plt.close()

    # Isquemia
    real = df_casos["isq_real_vol_ml"]
    pred = df_casos["isq_pred_vol_ml"]
    plt.figure(figsize=(7,7))
    plt.scatter(real, pred)
    min_v = min(real.min(), pred.min())
    max_v = max(real.max(), pred.max())
    plt.plot([min_v, max_v], [min_v, max_v])
    plt.xlabel("Volumen real isquemia (ml)")
    plt.ylabel("Volumen predicho IA (ml)")
    plt.title("Scatter plot: Isquemia")
    plt.tight_layout()
    path2 = os.path.join(output_dir, "scatter_isquemia.png")
    plt.savefig(path2, dpi=200, bbox_inches="tight")
    plt.close()

    return path1, path2

def guardar_grafica_dice():
    cases = df_casos["case"]
    x = np.arange(len(cases))

    plt.figure(figsize=(14,5))
    plt.plot(x, df_casos["brain_dice_3d"], marker="o", label="Dice cerebro")
    plt.plot(x, df_casos["isq_dice_3d"], marker="s", label="Dice isquemia")
    plt.xticks(x, cases, rotation=45)
    plt.ylabel("Dice score")
    plt.ylim(0,1)
    plt.title("Dice score por paciente")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(output_dir, "dice_pacientes.png")
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

    return path

def guardar_grafica_bland_altman():
    fig, axes = plt.subplots(1, 2, figsize=(14,6))

    # Cerebro
    real = df_casos["brain_real_vol_ml"]
    pred = df_casos["brain_pred_vol_ml"]
    mean_vals = (real + pred) / 2
    diff_vals = pred - real
    mean_diff = np.mean(diff_vals)
    std_diff = np.std(diff_vals)

    axes[0].scatter(mean_vals, diff_vals)
    axes[0].axhline(mean_diff, linestyle="--", label="Sesgo medio")
    axes[0].axhline(mean_diff + 1.96*std_diff, linestyle=":")
    axes[0].axhline(mean_diff - 1.96*std_diff, linestyle=":")
    axes[0].set_title("Bland-Altman Cerebro")
    axes[0].set_xlabel("Media de volumen (ml)")
    axes[0].set_ylabel("Diferencia IA - Real (ml)")
    axes[0].legend()

    # Isquemia
    real = df_casos["isq_real_vol_ml"]
    pred = df_casos["isq_pred_vol_ml"]
    mean_vals = (real + pred) / 2
    diff_vals = pred - real
    mean_diff = np.mean(diff_vals)
    std_diff = np.std(diff_vals)

    axes[1].scatter(mean_vals, diff_vals)
    axes[1].axhline(mean_diff, linestyle="--", label="Sesgo medio")
    axes[1].axhline(mean_diff + 1.96*std_diff, linestyle=":")
    axes[1].axhline(mean_diff - 1.96*std_diff, linestyle=":")
    axes[1].set_title("Bland-Altman Isquemia")
    axes[1].set_xlabel("Media de volumen (ml)")
    axes[1].set_ylabel("Diferencia IA - Real (ml)")
    axes[1].legend()

    plt.tight_layout()
    path = os.path.join(output_dir, "bland_altman.png")
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

    return path

def guardar_grafica_boxplots():
    fig, axes = plt.subplots(1, 2, figsize=(12,6))

    axes[0].boxplot([df_casos["brain_dice_3d"], df_casos["isq_dice_3d"]])
    axes[0].set_xticks([1,2])
    axes[0].set_xticklabels(["Cerebro", "Isquemia"])
    axes[0].set_ylabel("Dice score")
    axes[0].set_title("Distribución del Dice")

    axes[1].boxplot([df_casos["brain_rel_error_pct"].dropna(), df_casos["isq_rel_error_pct"].dropna()])
    axes[1].set_xticks([1,2])
    axes[1].set_xticklabels(["Cerebro", "Isquemia"])
    axes[1].set_ylabel("Error relativo (%)")
    axes[1].set_title("Distribución del error volumétrico")

    plt.tight_layout()
    path = os.path.join(output_dir, "boxplots_metricas.png")
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

    return path

def guardar_grafica_histogramas():
    fig, axes = plt.subplots(2, 2, figsize=(12,10))

    axes[0,0].hist(df_casos["brain_dice_3d"], bins=10)
    axes[0,0].set_title("Histograma Dice - Cerebro")
    axes[0,0].set_xlabel("Dice")
    axes[0,0].set_ylabel("Frecuencia")

    axes[0,1].hist(df_casos["isq_dice_3d"], bins=10)
    axes[0,1].set_title("Histograma Dice - Isquemia")
    axes[0,1].set_xlabel("Dice")
    axes[0,1].set_ylabel("Frecuencia")

    axes[1,0].hist(df_casos["brain_rel_error_pct"].dropna(), bins=10)
    axes[1,0].set_title("Histograma Error % - Cerebro")
    axes[1,0].set_xlabel("Error relativo (%)")
    axes[1,0].set_ylabel("Frecuencia")

    axes[1,1].hist(df_casos["isq_rel_error_pct"].dropna(), bins=10)
    axes[1,1].set_title("Histograma Error % - Isquemia")
    axes[1,1].set_xlabel("Error relativo (%)")
    axes[1,1].set_ylabel("Frecuencia")

    plt.tight_layout()
    path = os.path.join(output_dir, "histogramas_metricas.png")
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

    return path

img_vol_brain, img_vol_isq = guardar_grafica_volumenes()
img_scatter_brain, img_scatter_isq = guardar_grafica_scatter()
img_dice = guardar_grafica_dice()
img_bland = guardar_grafica_bland_altman()
img_box = guardar_grafica_boxplots()
img_hist = guardar_grafica_histogramas()

# 7) EXPORTAR DATAFRAMES A EXCEL
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="Resumen", index=False)
    df_casos.to_excel(writer, sheet_name="Datos_por_caso", index=False)
    df_slices.to_excel(writer, sheet_name="Datos_por_slice", index=False)

    # hojas vacías para meter gráficas
    pd.DataFrame({"Info": ["Gráficas del análisis"]}).to_excel(writer, sheet_name="Graficas_1", index=False)
    pd.DataFrame({"Info": ["Más gráficas del análisis"]}).to_excel(writer, sheet_name="Graficas_2", index=False)

# 8) FORMATEAR EXCEL
wb = load_workbook(excel_path)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
sub_fill = PatternFill("solid", fgColor="D9EAF7")
bold_font = Font(bold=True)
thin = Side(style="thin", color="D9D9D9")

for ws in wb.worksheets:
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    
    ws.freeze_panes = "A2"

    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                max_len = max(max_len, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = min(max_len + 2, 35)

# hoja resumen más legible
ws = wb["Resumen"]
ws["D2"] = "Resumen ejecutivo"
ws["D2"].font = Font(bold=True, size=14)
ws["D3"] = conclusion_global
ws["D3"].alignment = Alignment(wrap_text=True, vertical="top")
ws["D3"].fill = sub_fill
ws["D3"].border = Border(left=thin, right=thin, top=thin, bottom=thin)
ws.column_dimensions["D"].width = 65
ws.row_dimensions[3].height = 70

# formatos numéricos
for ws_name in ["Resumen", "Datos_por_caso", "Datos_por_slice"]:
    ws = wb[ws_name]
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if isinstance(cell.value, (int, float)):
                if "pct" in str(ws.cell(1, cell.column).value).lower() or "%" in str(ws.cell(1, cell.column).value):
                    cell.number_format = '0.00'
                else:
                    cell.number_format = '0.000'

# 9) INSERTAR GRÁFICAS EN EL EXCEL
ws1 = wb["Graficas_1"]
ws1["A1"] = "Gráficas principales del análisis"
ws1["A1"].font = Font(bold=True, size=14)

img = XLImage(img_vol_brain)
img.width = 700
img.height = 300
ws1.add_image(img, "A3")

img = XLImage(img_vol_isq)
img.width = 700
img.height = 300
ws1.add_image(img, "A22")

img = XLImage(img_scatter_brain)
img.width = 420
img.height = 420
ws1.add_image(img, "J3")

img = XLImage(img_scatter_isq)
img.width = 420
img.height = 420
ws1.add_image(img, "J25")

ws2 = wb["Graficas_2"]
ws2["A1"] = "Métricas de rendimiento"
ws2["A1"].font = Font(bold=True, size=14)

img = XLImage(img_dice)
img.width = 700
img.height = 260
ws2.add_image(img, "A3")

img = XLImage(img_bland)
img.width = 700
img.height = 300
ws2.add_image(img, "A18")

img = XLImage(img_box)
img.width = 620
img.height = 300
ws2.add_image(img, "A38")

img = XLImage(img_hist)
img.width = 620
img.height = 520
ws2.add_image(img, "J3")

wb.save(excel_path)

print(f"\n✅ Informe Excel generado en:\n{excel_path}")

In [ ]:

CASOS_INTERES = df_casos["case"].head(3).values  # Primeros 3 casos

for caso_name in CASOS_INTERES:
    print(f"\n{'='*70}")
    print(f"ANÁLISIS DETALLADO: {caso_name}")
    print(f"{'='*70}")
    
    df_case = df_slices[df_slices["case"] == caso_name]
    
    case_dir = Path(DATASET_DIR) / caso_name
    nifti_path = case_dir / "Seq No.nii"
    nii, vol = cargar_volumen_nifti(nifti_path)
    
    cerebro_dir = case_dir / "cerebro"
    isq_dir = case_dir / "isquemia"
    
    # Mostrar 3 slices: inicio, medio, fin
    slices_a_mostrar = [0, len(df_case)//2, len(df_case)-1]
    etiquetas = ["Inicio", "Medio", "Fin"]
    
    for sub_idx, (row_idx, etiqueta) in enumerate(zip(slices_a_mostrar, etiquetas)):
        row = df_case.iloc[row_idx]
        slice_idx = int(row["slice_idx_original"])
        
        img = vol[:, :, slice_idx]
        
        # Cargar máscaras reales
        cerebro_real = np.zeros_like(img)
        isq_real = np.zeros_like(img)
        
        for f in cerebro_dir.glob("*.png"):
            if int(f.name.split("-")[0]) - 1 == slice_idx:
                cerebro_real = load_mask(f)
        
        for f in isq_dir.glob("*.png"):
            if int(f.name.split("-")[0]) - 1 == slice_idx:
                isq_real = load_mask(f)
        
        cerebro_real = resize_mask(cerebro_real)
        isq_real = resize_mask(isq_real)
        
        # Predicciones
        img_resized = resize_img_slice(img)
        input_img = img_resized[np.newaxis, :, :, np.newaxis]
        
        brain_pred = model_brain.predict(input_img, verbose=0)[0,:,:,0]
        brain_pred = (brain_pred > BRAIN_THRESHOLD).astype(np.uint8)
        
        input_brain = input_img * brain_pred[np.newaxis,:,:,np.newaxis]
        isq_pred = model_isq.predict(input_brain, verbose=0)[0,:,:,0]
        isq_pred = (isq_pred > ISQ_THRESHOLD).astype(np.uint8)
        
        # Métricas
        brain_dice = dice_numpy(cerebro_real, brain_pred)
        isq_dice = dice_numpy(isq_real, isq_pred)
        brain_iou = iou_numpy(cerebro_real, brain_pred)
        isq_iou = iou_numpy(isq_real, isq_pred)
        
        # Crear figura con comparación lado a lado
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        
        # Fila 1: Ground Truth
        # GT - MRI + Cerebro
        axes[0, 0].imshow(img_resized, cmap="gray")
        axes[0, 0].imshow(cerebro_real, cmap="Greens", alpha=0.5)
        axes[0, 0].set_title("GT: Cerebro", fontsize=12, fontweight='bold')
        axes[0, 0].axis("off")
        
        # GT - MRI + Isquemia
        axes[0, 1].imshow(img_resized, cmap="gray")
        axes[0, 1].imshow(isq_real, cmap="Reds", alpha=0.5)
        axes[0, 1].set_title("GT: Isquemia", fontsize=12, fontweight='bold')
        axes[0, 1].axis("off")
        
        # GT - Ambas mascaras
        axes[0, 2].imshow(img_resized, cmap="gray")
        axes[0, 2].imshow(cerebro_real, cmap="Greens", alpha=0.5)
        axes[0, 2].imshow(isq_real, cmap="Reds", alpha=0.5)
        axes[0, 2].set_title("GT: Cerebro + Isquemia", fontsize=12, fontweight='bold')
        axes[0, 2].axis("off")
        
        # Fila 2: Predicciones
        # Pred - MRI + Cerebro
        axes[1, 0].imshow(img_resized, cmap="gray")
        axes[1, 0].imshow(brain_pred, cmap="Greens", alpha=0.5)
        texto = f"Cerebro\nDice: {brain_dice:.3f}\nIoU: {brain_iou:.3f}"
        axes[1, 0].set_title(f"Predicción: {texto}", fontsize=11, fontweight='bold')
        axes[1, 0].axis("off")
        
        # Pred - MRI + Isquemia
        axes[1, 1].imshow(img_resized, cmap="gray")
        axes[1, 1].imshow(isq_pred, cmap="Reds", alpha=0.5)
        texto = f"Isquemia\nDice: {isq_dice:.3f}\nIoU: {isq_iou:.3f}"
        axes[1, 1].set_title(f"Predicción: {texto}", fontsize=11, fontweight='bold')
        axes[1, 1].axis("off")
        
        # Pred - Ambas mascaras
        axes[1, 2].imshow(img_resized, cmap="gray")
        axes[1, 2].imshow(brain_pred, cmap="Greens", alpha=0.5)
        axes[1, 2].imshow(isq_pred, cmap="Reds", alpha=0.5)
        axes[1, 2].set_title("Predicción: Cerebro + Isquemia", fontsize=12, fontweight='bold')
        axes[1, 2].axis("off")
        
        fig.suptitle(f"{caso_name} - Slice {etiqueta} (índice {slice_idx})", 
                     fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.show()

print(f"\n✅ Análisis detallado completado para {len(CASOS_INTERES)} casos")